# Source-Only Feature Selection for Phishing URL Detection — Bidirectional Cross-Dataset Evaluation

Notebook ini adalah **versi naratif** dari pipeline riset di `docs/PRD.md`. Seluruh logika ada di paket
`src/phishresearch/` (diuji 61 tes); notebook hanya **memanggil** modul, menampilkan artefak, dan memverifikasi ulang.

| RQ | Pertanyaan |
|---|---|
| RQ1 | Seberapa jauh RF dan LightGBM turun saat diuji pada sumber lain dengan domain terpisah? |
| RQ2 | Apakah STABLE_PI mengubah performa transfer dibanding MEAN_PI pada *k* sama dan ALL? |
| RQ3 | Apakah pengurangan fitur menurunkan biaya ekstraksi/inferensi secara terukur? |
| RQ4 | Fitur apa yang diandalkan model, dan bagaimana ranking atribusinya berubah dari sumber ke target? |

**Aturan eksekusi.** Tahap mahal atau yang terkunci freeze (`develop`, `freeze`, `evaluate`, `analyze`) hanya dijalankan
bila artefaknya belum ada; jika sudah ada, notebook memuatnya dan **memverifikasi hash freeze**. Tidak ada sel yang
mengubah data, split, model, atau threshold yang sudah dibekukan.

## 0. Setup

In [ ]:
import os, sys, json, warnings
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "configs" / "protocol.yaml").exists():   # notebook dijalankan dari notebooks/
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from phishresearch.common import Paths, load_yaml, read_json, effective_threads, environment_info
from phishresearch.reporting import COLORS, HATCH, INK2, _style

CFG = load_yaml("configs/protocol.yaml")
REGISTRY = load_yaml("configs/datasets.yaml")
P = Paths.from_config(CFG, ROOT)
_style(plt); plt.rcParams.update({"font.size": 9, "figure.dpi": 110})
pd.set_option("display.precision", 4, "display.max_colwidth", 80)

env = environment_info()
print(f"root      : {ROOT}\npython    : {env['python']}  |  CPU logical {env['cpu_count_logical']}  |  RAM {env['ram_gb']} GB")
print("packages  :", ", ".join(f"{k} {v}" for k, v in env["packages"].items()))
print(f"threads   : {effective_threads(CFG)}  |  seeds {CFG['models']['seeds']}  |  k candidates {CFG['selection']['k_candidates']}")

In [ ]:
def exists(p) -> bool:
    return Path(p).exists()

def stage(name: str, artefact: Path, fn):
    # Run fn() only when the artefact is missing; never overwrite frozen artefacts.
    if exists(artefact):
        print(f"[{name}] artefact exists -> loaded ({artefact.relative_to(ROOT)})")
    else:
        print(f"[{name}] running ..."); fn(); print(f"[{name}] done")

def fmt_pct(x): return f"{100*x:.1f}%"

## 1. Pengadaan data (data gate)
Dua sumber dipin ke commit GitHub: **A = EBBU-2017** (JSON per kelas) dan **B = HISPAR–Phishstats (HP)**
(gabungan `hp_train/valid/test`). File diunduh ke temp → divalidasi (bukan HTML, ukuran = tree GitHub) → *atomic rename*;
SHA256 disimpan di manifest. Salinan `ebbu_*.csv` di repo B hanya dipakai untuk memverifikasi konvensi label B.

In [ ]:
from phishresearch.acquire import acquire
stage("acquire", P.raw / "manifest.json", lambda: acquire(Path("configs/datasets.yaml"), P.raw))
man = read_json(P.raw / "manifest.json")["files"]
pd.DataFrame([{ "file": k, "role": v["role"], "commit": v["commit"][:8], "bytes": v["size"],
                "sha256": v["sha256"][:16] + "…", "expected": v["expected_records"], "observed": v["observed_records"]}
              for k, v in man.items()])

## 2. Audit & cohort domain-disjoint
Urutan (PRD §9): parse → dedup dalam sumber → karantina konflik label → audit overlap lintas sumber →
**purge simetris** semua registered domain (PSL ICANN, offline) yang muncul di kedua sumber.
Setiap URL yang dibuang punya tepat satu alasan primer sehingga attrition dapat direkonsiliasi.

In [ ]:
from phishresearch.prepare import run_audit
stage("audit", P.processed / "records.parquet", lambda: run_audit(CFG, REGISTRY, P))
audit = read_json(P.artifacts / "audit.json")

rows = []
for s, name in (("A", "EBBU"), ("B", "HP")):
    for st, v in audit["sources"][s]["stages"].items():
        rows.append({"source": name, "stage": st, **v})
attrition = pd.DataFrame(rows)
display(attrition.pivot_table(index="stage", columns="source", values=["n", "benign", "phishing"], sort=False))

lm = audit["label_mapping_check"]
print(f"Label mapping check : {lm['joined_rows']:,} URL di-join, agreement = {fmt_pct(lm['agreement'])} -> {lm['conclusion']}")
print(f"Shared domains purged: {audit['cross_source']['shared_groups']:,}  |  exact URL overlap: {audit['cross_source']['exact_url_key_overlap_records']}")
print(f"Group intersection setelah purge: {audit['cross_source']['cohort_group_intersection']} (harus 0)")
print("Rekonsiliasi:", {s: audit['sources'][s]['reconciles'] for s in 'AB'}, "| warnings:", audit["warnings"] or "none")

In [ ]:
# Verifikasi ulang langsung dari records.parquet (bukan dari audit.json)
rec = pd.read_parquet(P.processed / "records.parquet")
coh = rec[rec.in_cohort]
assert not set(coh[coh.dataset_id == "A"].group_id) & set(coh[coh.dataset_id == "B"].group_id)
display(rec.groupby(["dataset_id", rec.reason.fillna("in_cohort")]).size().unstack(0).fillna(0).astype(int))

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, s in zip(axes, "AB"):
    top = pd.Series(audit["sources"][s]["groups_cohort"]["top_groups"]).head(8)[::-1]
    ax.barh(top.index, top.values, color=COLORS["ALL"]); ax.set_title(f"Domain terbesar di cohort {s}")
    ax.set_xlabel("URL"); ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

## 3. Split group-disjoint FIT / VAL / TEST
Penempatan domain utuh secara deterministik (seed 20261115) ke partisi dengan kuota kelas paling kosong.
**Loader development menolak TEST**; label TEST disimpan di file terpisah yang hanya dibuka `evaluation.py` setelah freeze.

In [ ]:
from phishresearch.prepare import run_prepare, load_dev
from phishresearch.common import GateError
stage("prepare", P.processed / "dev_features.parquet", lambda: run_prepare(CFG, P))
prep = read_json(P.artifacts / "prepare.json")
display(pd.DataFrame(prep["partitions"]).T.assign(frac=lambda d: d.frac.map(fmt_pct), prevalence=lambda d: d.prevalence.map(fmt_pct)))

sp = pd.read_parquet(P.processed / "splits.parquet")
for s, d in sp.groupby("dataset_id"):
    g = d.groupby("partition").group_id.apply(set)
    assert not (g["FIT"] & g["VAL"] or g["FIT"] & g["TEST"] or g["VAL"] & g["TEST"])
print("Partisi group-disjoint: OK")
try:
    load_dev(P, "A", "TEST")
except GateError as e:
    print("Loader policy:", e)

## 4. Fitur leksikal (37) — demo pada domain reserved
Setiap fitur adalah fungsi terpisah sehingga subset hanya menghitung yang diperlukan (*dependency pruning*);
parsing selalu dibayar. Contoh di bawah memakai domain dokumentasi/reserved — tidak ada URL dataset yang dibuka.

In [ ]:
from phishresearch.features import extract_features, FEATURE_IDS, FEATURE_NAMES
from phishresearch.urls import canonicalize_url

demo = ["http://login.secure.example.com:8080/a_b/c-d?x=1&y=%41#top",
        "example.org/blog/post",
        "https://bücher.example/verify-account",
        "http://192.0.2.10/wallet/update.php?id=7",
        "javascript:alert(1)"]
for u in demo:
    p = canonicalize_url(u)
    print(f"{u[:55]:55s} -> {type(p).__name__}: {getattr(p, 'group_key', getattr(p, 'reason', ''))}")

X_demo = extract_features(demo[:4])
X_demo.columns = [f"{c} {FEATURE_NAMES[c]}" for c in X_demo.columns]
display(X_demo.T)

sub = ["F02", "F20", "F36"]
assert np.array_equal(extract_features(demo[:4], sub).to_numpy(), extract_features(demo[:4])[sub].to_numpy())
print("Subset extraction == proyeksi ALL: OK")

## 5. Development source-only (nested selection)
Per sumber dan seed: 3 outer domain fold di FIT → PI dihitung **hanya** dari outer-train → classifier dilatih pada top-*k*
→ skor AP outer hold-out → pilih *k* terkecil dalam 0,01 AP dari terbaik → PI ulang di seluruh FIT → model final
(ALL, MEAN_PI, STABLE_PI, MEAN_PI@*k* STABLE) → threshold dikunci di source VAL. TEST tidak pernah disentuh.

In [ ]:
from phishresearch.develop import develop_source
for s in "AB":
    stage(f"develop {s}", P.dev / s / "develop.json",
          lambda s=s: develop_source(s, load_dev(P, s, "FIT"), load_dev(P, s, "VAL"), CFG, effective_threads(CFG), P.dev / s))

dev = {s: read_json(P.dev / s / "develop.json") for s in "AB"}
display(pd.DataFrame({(s, seed): v["chosen_k"] for s in "AB" for seed, v in dev[s]["seeds"].items()}).rename_axis("model/selector"))
for s in "AB":
    t = dev[s]["timing"]
    print(f"{s}: PI {t['pi_seconds']/60:.1f} min | outer fits {t['outer_fit_seconds']/60:.1f} min | "
          f"final fits {sum(t['final_fit_seconds'].values()):.0f} s | full-FIT selector hash ∉ outer hashes: "
          f"{all(v['full_selector_hash'] not in v['outer_selector_hashes'] for v in dev[s]['seeds'].values())}")

In [ ]:
# Kurva nested-CV: mean outer AP vs k (sumber hanya FIT)
fig, axes = plt.subplots(1, 4, figsize=(13, 2.8), sharey=False)
for ax, (s, m) in zip(axes, [(s, m) for s in "AB" for m in ("RF", "LGBM")]):
    cv = pd.read_csv(P.dev / s / "outer_cv.csv")
    g = cv[cv.model == m].groupby(["selector", "k"]).ap.agg(["mean", "std"]).reset_index()
    for sel, d in g.groupby("selector"):
        ax.errorbar(d.k, d["mean"], yerr=d["std"], marker="o", ms=4, lw=1.5, capsize=2, color=COLORS[sel], label=sel)
    ax.set_title(f"{s} · {m}"); ax.set_xlabel("k"); ax.set_xticks(CFG["selection"]["k_candidates"])
axes[0].set_ylabel("outer hold-out AP"); axes[0].legend(fontsize=7)
plt.tight_layout(); plt.show()

In [ ]:
# Permutation importance di full FIT (rata-rata seed) + votes STABLE_PI
from phishresearch.selection import PiResult, rank_features

def pi_table(s):
    # mean PI and STABLE_PI votes per feature, averaged over seeds (computed by the package's ranking code)
    rows = []
    for seed in sorted(dev[s]["seeds"]):
        pj = read_json(P.dev / s / "selectors" / f"pi_s{seed}_full.json")
        pi = PiResult(pj["feature_ids"], np.array(pj["pi_folds"]), np.array(pj["pi_repeats"]), pj["fit_ids_hash"], pj["n_rows"])
        _, st = rank_features(pi, "STABLE_PI", CFG["selection"]["q"])
        rows.append(pd.DataFrame({"mean_pi": st["mean_pi"], "votes": st["votes"]}))
    t = sum(rows) / len(rows)
    t.index = [f"{f} {FEATURE_NAMES[f]}" for f in t.index]
    return t

fig, axes = plt.subplots(1, 2, figsize=(11, 8), sharey=True)
for ax, s in zip(axes, "AB"):
    t = pi_table(s)
    ax.barh(t.index, t.mean_pi, color=[COLORS["STABLE_PI"] if v >= 2.5 else COLORS["ALL"] for v in t.votes])
    ax.axvline(0, color=INK2, lw=.6); ax.set_title(f"Sumber {s}: mean PI (ΔAP); hijau = votes ≥ 3"); ax.grid(axis="y", visible=False)
axes[0].invert_yaxis(); plt.tight_layout(); plt.show()

## 6. Freeze — kunci kedua arah sebelum TEST dibuka
Manifest meng-hash config, kode (kecuali presentasi), PSL, records, split, fitur, file TEST, dan **semua bundle** kedua arah.
Sel berikut memverifikasi ulang seluruh hash dan mendemonstrasikan bahwa perubahan satu byte ditolak.

In [ ]:
from phishresearch.freeze import build_freeze, verify_freeze
stage("freeze", P.freeze, lambda: build_freeze(Path("configs/protocol.yaml"), Path("configs/datasets.yaml"), P,
                                                "notebook run (author review pending)", CFG))
fr = verify_freeze(P.freeze, ROOT)
print(f"freeze_hash : {fr['freeze_hash']}\napproved_by : {fr['approved_by']}\ncreated_at  : {fr['created_at']}")
print(f"files hashed: {len(fr['body']['file_hashes'])} | bundles in plan: {len(fr['body']['plan'])} | VERIFIED")

# Tamper demo pada SALINAN manifest (file asli tidak disentuh)
import copy, tempfile
bad = copy.deepcopy(fr); bad["body"]["seeds"] = [1]
with tempfile.TemporaryDirectory() as td:
    fp = Path(td) / "freeze.json"; fp.write_text(json.dumps(bad))
    try: verify_freeze(fp, ROOT)
    except GateError as e: print("Tamper demo  : rejected ->", e)

## 7. Evaluasi final (satu batch)
Setiap bundle beku dinilai pada TEST sumbernya (internal) dan TEST sumber lain (cross) dengan **threshold yang sama dari source VAL**.
Kemudian semua metrik **dihitung ulang** dari `predictions.parquet` (cek I06).

In [ ]:
from phishresearch.evaluation import run_evaluation
from phishresearch.analysis import recompute_check
stage("evaluate", P.eval / "metrics.json", lambda: run_evaluation(P.freeze, P))
ev = read_json(P.eval / "metrics.json")
assert ev["freeze_hash"] == fr["freeze_hash"], "evaluation belongs to another freeze"
PRED = pd.read_parquet(P.eval / "predictions.parquet")
MET = pd.DataFrame(ev["metrics"])
print(f"run_id {ev['run_id']} | {len(PRED):,} prediction rows | {len(MET)} metric rows | status {ev['status']}")
print("I06 recompute:", recompute_check(PRED, MET))

## 8. Analisis — CI, kontras, biaya, SHAP
`analyze` menghasilkan ringkasan per konfigurasi, cluster-bootstrap CI (domain di-resample), empat kontras primer
STABLE_PI vs MEAN_PI (*k* sama), benchmark latensi, dan SHAP. Dijalankan hanya bila belum ada.

In [ ]:
from phishresearch.analysis import run_analysis
stage("analyze", P.analysis / "results_registry.json", lambda: run_analysis(P.freeze, CFG, P))
A = P.analysis
SUM = pd.read_csv(A / "summary.csv", dtype={"k_per_seed": str})
CI = pd.read_csv(A / "config_ci.csv")
CON = read_json(A / "contrasts.json")
print(read_json(A / "results_registry.json")["status"])

### 8.1 RQ1 — performa internal vs lintas dataset

In [ ]:
main = SUM[SUM.policy == "f1_max"].merge(CI, on=["source", "target", "model", "config"])
view = main[main.config.isin(["ALL", "MEAN_PI@budget", "STABLE_PI@budget"])]
tbl = view.assign(cell=view.source + "→" + view.target,
                  F1=view.f1_mean.map("{:.3f}".format) + " [" + view.ci_low.map("{:.3f}".format) + ", " + view.ci_high.map("{:.3f}".format) + "]")
display(tbl.pivot_table(index=["cell", "model"], columns="config", values="F1", aggfunc="first", sort=False))

fig, axes = plt.subplots(1, 4, figsize=(13, 3), sharey=True)
for ax, (s, t, ttl) in zip(axes, [("A","A","EBBU internal"),("A","B","EBBU → HP"),("B","B","HP internal"),("B","A","HP → EBBU")]):
    for i, m in enumerate(("RF", "LGBM")):
        for j, conf in enumerate(("ALL", "MEAN_PI@budget", "STABLE_PI@budget")):
            r = main[(main.source == s) & (main.target == t) & (main.model == m) & (main.config == conf)].iloc[0]
            fam = conf.split("@")[0]; x = i + (j - 1) * .27
            ax.bar(x, r.f1_mean, .25, color=COLORS[fam], hatch=HATCH[fam], edgecolor="white", label=fam if (i == 0 and s == t == "A") else None)
            ax.errorbar(x, r.f1_mean, [[r.f1_mean - r.ci_low], [r.ci_high - r.f1_mean]], fmt="none", ecolor="k", lw=.8, capsize=2)
    ax.set_title(ttl); ax.set_xticks([0, 1], ["RF", "LightGBM"]); ax.set_ylim(0, 1); ax.grid(axis="x", visible=False)
axes[0].set_ylabel("F1 phishing (TEST)"); fig.legend(loc="upper center", ncol=3, bbox_to_anchor=(.5, 1.06))
plt.tight_layout(); plt.show()

TD = pd.read_csv(A / "transfer_drop.csv")
display(TD.groupby(["source", "model", "config"]).drop_pp.agg(["mean", "std"]).round(1).rename(columns={"mean": "drop (pp)", "std": "sd seeds"}))

### 8.2 RQ2 — kontras primer STABLE_PI vs MEAN_PI pada *k* sama (target TEST)

In [ ]:
def contrast_frame(items):
    return pd.DataFrame([{ "cell": f"{c['source']}→{c['target']}", "model": c["model"], "contrast": f"{c['a']} − {c['b']}",
        "k (a | b)": f"{c['k_a']} | {c['k_b']}", "ΔF1": c["delta"], "CI low": c["ci_low"], "CI high": c["ci_high"],
        "per seed": np.round(c["per_seed_delta"], 4).tolist(), "same model": all(c["identical_bundle"]),
        "|Δ| ≥ 0.01": abs(c["delta"]) >= CFG["effects"]["f1_abs"]} for c in items])

PRIM = contrast_frame(CON["primary"]); display(PRIM)
fig, ax = plt.subplots(figsize=(6, 2.6))
for i, r in PRIM.iterrows():
    ax.errorbar(r["ΔF1"], i, xerr=[[r["ΔF1"] - r["CI low"]], [r["CI high"] - r["ΔF1"]]], fmt="o", color=COLORS["STABLE_PI"], capsize=3)
ax.axvline(0, color=INK2, lw=.8); ax.axvspan(-.01, .01, color="#eeeeea", zorder=0, label="|Δ| < 0.01")
ax.set_yticks(range(len(PRIM)), PRIM.cell + " " + PRIM.model); ax.invert_yaxis(); ax.set_xlabel("ΔF1 target (STABLE − MEAN), 95% cluster-bootstrap CI")
ax.legend(fontsize=7, loc="lower right"); plt.tight_layout(); plt.show()
display(Markdown("**Sekunder (lintas dataset):**")); display(contrast_frame([c for c in CON["secondary"] if c["source"] != c["target"]]))

### 8.3 RQ3 — biaya komputasi

In [ ]:
B = pd.read_csv(A / "benchmark.csv")
cost = B[B.stage.isin(["extract", "predict", "end_to_end"])].pivot_table(
    index=["source", "model", "selector", "k"], columns=["threads", "batch", "stage"], values="median_us_per_url")
display(cost.round(1))
rel = B[(B.stage == "end_to_end")].pivot_table(index=["source", "model", "selector", "k"], columns=["threads", "batch"], values="rel_to_all")
display(Markdown("**Latensi end-to-end relatif terhadap ALL** (< 0.90 = penurunan ≥ 10%, ambang praktis PRD):")); display(rel.round(2))
bj = read_json(A / "benchmark.json")
display(pd.DataFrame([{ "bundle": b["bundle_id"], "MB": b["model_file_bytes"] / 2**20, "load ms": b["load_ms"], "fit s": b["fit_seconds"]} for b in bj["bundles"]]).round(2))

### 8.4 RQ4 — atribusi SHAP (model asli, interventional, output probabilitas)

In [ ]:
SS = pd.read_csv(A / "shap_summary.csv"); SC = pd.read_csv(A / "shap_compare.csv")
print("max additivity error:", f"{SS.additivity_max_abs_err.max():.1e}")
display(SC.round(3))

import shap
cols = sorted(f for f in os.listdir(A / "shap") if "-ALL-k37-s17_source" in f)
for f in cols[:4]:
    z = np.load(A / "shap" / f, allow_pickle=True)
    names = [f"{i} {FEATURE_NAMES[i]}" for i in z["ids"]]
    plt.figure(); shap.summary_plot(z["shap"], z["X"], feature_names=names, max_display=10, show=False, plot_size=(7, 3.2))
    plt.title(f.replace("_source.npz", " — source TEST"), fontsize=9); plt.tight_layout(); plt.show()

## 9. Jawaban RQ (dihasilkan dari artefak — tanpa angka diketik manual)

In [ ]:
def f1(s, t, m, c): return main[(main.source == s) & (main.target == t) & (main.model == m) & (main.config == c)].iloc[0]
lines = ["**RQ1 — penurunan transfer (ALL, F1 threshold source-VAL):**"]
for s in "AB":
    t = "B" if s == "A" else "A"
    for m in ("RF", "LGBM"):
        a, b = f1(s, s, m, "ALL"), f1(s, t, m, "ALL")
        lines.append(f"- {s}→{t} {m}: internal {a.f1_mean:.3f} → cross {b.f1_mean:.3f} (drop {100*(a.f1_mean-b.f1_mean):.1f} pp)")
lines.append("\n**RQ2 — STABLE_PI vs MEAN_PI (k sama, target TEST):**")
for _, r in PRIM.iterrows():
    verdict = ("CI > 0" if r["CI low"] > 0 else "CI < 0" if r["CI high"] < 0 else "CI mencakup 0")
    lines.append(f"- {r.cell} {r.model}: ΔF1 = {r['ΔF1']:+.4f} [{r['CI low']:+.4f}, {r['CI high']:+.4f}] → {verdict}"
                 f"{' (model identik: subset sama)' if r['same model'] else ''}")
e2e = B[(B.stage == "end_to_end") & (B.selector != "ALL")]
lines.append(f"\n**RQ3 — latensi end-to-end relatif ALL:** batch-1/1-thread {e2e[(e2e.threads==1)&(e2e.batch=='batch1')].rel_to_all.min():.2f}–"
             f"{e2e[(e2e.threads==1)&(e2e.batch=='batch1')].rel_to_all.max():.2f}; batch-1024/4-thread "
             f"{e2e[(e2e.threads==4)&(e2e.batch=='batch1024')].rel_to_all.min():.2f}–{e2e[(e2e.threads==4)&(e2e.batch=='batch1024')].rel_to_all.max():.2f}.")
st = SC[SC.kind == "source_vs_target"]
lines.append(f"\n**RQ4 — kesamaan ranking SHAP sumber vs target:** Spearman {st.spearman.min():.2f}–{st.spearman.max():.2f}; "
             f"top-10 Jaccard {st.top10_jaccard.min():.2f}–{st.top10_jaccard.max():.2f}. (Perubahan atribusi ≠ kausalitas.)")
display(Markdown("\n".join(lines)))

---
**Batas klaim.** Benchmark terkurasi domain-disjoint dari dua dataset publik lama; CI kondisional pada split & seed tetap;
loader TEST + freeze adalah kontrol prosedural. Manuskrip IEEE (`manuscript/main.pdf`) memakai angka yang sama melalui
`manuscript/generated/macros.tex`.